# **Pipeline Unificado — Extracción, Clasificación y Anonimización de Tweets**
### Arquitectura v3 · Por usuario · Todo en memoria · RGPD compliant
---
**Flujo:**
1. **Stage 1** — Extracción de tweets por keywords (2020-2025, API twitterapi.io)
2. **Stage 2** — LLM_UserType: Clasificación Org/Indiv con **Gemma 3 12B** local
3. **Stage 3** — Descarga de hasta 100 tweets por usuario 
4. **Stage 4** — Anonimización multicapa: Regex → Presidio+BERT → **LLaMA 3 8B** selectivo → Hash IDs
5. **Stage 5** — Generalización semántica de `profile_bio` con **Gemma 3 12B** + traducción ES→FR

**Garantías RGPD:** los datos crudos nunca tocan disco; solo el output ya anonimizado puede persistir.


In [ ]:
!pip install -q -U "transformers>=4.50.0" accelerate bitsandbytes
!pip install -q "presidio-analyzer[transformers]" presidio-anonymizer
!pip install -q spacy
!python -m spacy download es_core_news_sm -q
!pip install -q langdetect sentencepiece sacremoses tqdm requests
!pip install -q ollama
# Dependencia del instalador de Ollama
!apt-get install -q -y zstd
# Instalar servidor Ollama (necesario para LLaMA 3 en Fase B)
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Dependencias instaladas.")


In [ ]:
import secrets

# API 
TWITTER_API_KEY = ""

# Modelos
GEMMA_MODEL_ID = "unsloth/gemma-3-12b-it" # HuggingFace, sin token
LLAMA_OLLAMA_MODEL = "llama3" # Ollama — sin token HuggingFace
BERT_MODEL = "Davlan/bert-base-multilingual-cased-ner-hrl"
SPACY_MODEL = "es_core_news_sm"

# Extracción (Stage 1)
LANG = "es"
LAT = "40.4168"
LON = "-3.7038"
RADIUS = "600km"
START_YEAR = 2020
END_YEAR = 2025
KEYWORD_BATCH_SIZE = 12 # keywords por query (límite URL)
API_WAIT_TIME = 2 # segundos entre llamadas API

# Descarga por usuario (Stage 3)
MAX_TWEETS_USER = 100
MAX_WORKERS = 5 # hilos paralelos

# Anonimización (Stage 4)
PRESIDIO_SCORE_THRESHOLD = 0.4
LLM_ONLY_WITH_ENTITIES = True # LLaMA solo en tweets donde BERT detectó entidades

# Flags GPU 
SWAP_MODELS = True # True → T4 free (15 GB). False → A100 Pro (40 GB), sin swap

# Sesión / checkpoint
SESSION_OFFSET = 0 # Índice de usuario desde el que continuar (para reanudar)
USE_DRIVE_CHECKPOINT = True # Guarda output anonimizado en Drive tras cada usuario
CHECKPOINT_PATH = "/content/drive/MyDrive/TFM/pipeline_checkpoint.jsonl"

# Salt volátil RGPD
VOLATILE_SALT = secrets.token_hex(32)
print(f"🔐 Salt volátil: {VOLATILE_SALT[:12]}... (no persiste entre sesiones — correcto por RGPD)")

# Debug / prueba rápida
DEBUG_MODE = False # False = producción completa
MAX_USERS_DEBUG = None # None = todos los usuarios; 2 = modo prueba
DEBUG_SPECIFIC_DATES = [ # Fechas donde se confirma actividad
    "2023-03-28",
    "2022-11-22",
    "2020-12-23",
]
DEBUG_KEYWORDS = [ # Keywords presentes en esas fechas
    "tengo ansiedad",
    "no puedo parar de llorar",
    "sin ganas de vivir",
    "Suicida",
]

# Keywords (100 términos relacionados con suicidio en español)
KEYWORDS = [
    "a nadie le importa cómo estoy", "a nadie le importo",
    "A punto de acabar con mi vida", "A punto de colgarme",
    "A punto de matarme", "a punto de suicidarme",
    "autolesionarme", "carta de despedida", "cómo matarse",
    "cómo suicidarse", "cortes en las muñecas", "debería matarme",
    "deseando morirme", "despedirme de todos",
    "diagnosticada con depresión", "diagnosticado con depresion",
    "doy asco a la gente", "enfermedad mental crónica",
    "escribiendo mi carta", "Estoy a punto de hacerlo",
    "estoy cansada de existir", "estoy cansado de existir",
    "fuerzas para matarme", "fuerzas para suicidarme",
    "hacer sufrir a mi familia", "hago sufrir a todos",
    "he intentado cometer suicidio", "he intentado suicidarme",
    "He planeado mi muerte", "hoy acabaré con mi vida",
    "impulso suicida", "impulsos suicidas",
    "intenté ahogarme", "intenté asfixiarme",
    "intenté cometer suicidio", "intenté suicidarme",
    "Intento de suicidio", "intentos de suicidio",
    "listo para acabar con mi vida", "lloro cada noche",
    "me autolesioné", "me he autolesionado",
    "me odio a mi mismo", "me siento cada vez peor",
    "modos de suicidarse", "morir de sobredosis",
    "morir el día de mi cumpleaños", "muerte es la mejor opción",
    "muerte planificada", "necesito que alguien me escuche",
    "no consigo suicidarme", "no importo a nadie",
    "no me envies flores", "no puedo parar de llorar",
    "no puedo vivir con esto", "no quiero estar en este mundo",
    "no tengo ganas ni de vivir", "no vengáis a mi entierro",
    "no vengas a mi entierro", "nunca me he sentido tan mal",
    "pensamientos suicidas", "Pensando en acabar con mi vida",
    "pensando en matarme", "Pensando en suicidarme",
    "Por qué debería seguir vivo", "por qué seguir vivo",
    "probablemente estaré muerta", "Probablemente me suicidaré",
    "quiero suicidarme", "razón para seguir viviendo",
    "razones para matarme", "razones para morir",
    "razones para suicidarme", "sin ganas de vivir",
    "sólo quiero morir", "soy inútil para la sociedad",
    "Suicida", "suicidarse fácil", "suicidarse rápido",
    "suicidarse sin dolor", "suicidio es la mejor opción",
    "suicidio fácil", "suicidio inevitable", "suicidio planeado",
    "suicidio planificado", "suicidio rápido", "suicidio sin dolor",
    "tengo ansiedad", "Tengo planes de suicidarme",
    "terapia no funciona", "tirarme al tren", "voy a suicidarme",
    "#suicidio", "#DiaMundialPrevencionSuicidio",
    "#suicideprevention", "#alone", "#broken", "#death",
    "#suicide", "#sad", "#mentalhealth", "#suicidal",
]

print(f"⚙️ Configuración cargada: {len(KEYWORDS)} keywords · {START_YEAR}-{END_YEAR} · SWAP_MODELS={SWAP_MODELS}")

In [ ]:
# Imports
import os, re, gc, json, time, copy, hashlib, unicodedata
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import requests
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor, Gemma3ForConditionalGeneration,
    MarianMTModel, MarianTokenizer,
    BitsAndBytesConfig,
)
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import TransformersNlpEngine
from presidio_anonymizer import AnonymizerEngine
from langdetect import detect, DetectorFactory, LangDetectException
DetectorFactory.seed = 42

print("📦 Imports listos.")

# ═══════════════════════════════════════════════════════════════
# GESTIÓN DE MODELOS
# ═══════════════════════════════════════════════════════════════

def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

def unload_model(model, name="modelo"):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"♻️ {name} descargado de VRAM.")

def load_gemma():
    print(f"⬇️ Cargando {GEMMA_MODEL_ID} (4-bit)...")
    processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        GEMMA_MODEL_ID,
        quantization_config=_bnb_config(),
        device_map="auto",
        torch_dtype=torch.bfloat16,
    ).eval()
    print("✅ Gemma 3 12B listo.")
    return model, processor

def load_llama():
    """Verifica que Ollama esté listo y el modelo llama3 esté descargado."""
    import ollama as _ollama
    try:
        models = [m.model for m in _ollama.list().models]
        if not any(LLAMA_OLLAMA_MODEL in m for m in models):
            print(f"⬇️  Descargando {LLAMA_OLLAMA_MODEL} via Ollama (puede tardar varios minutos)...")
            _ollama.pull(LLAMA_OLLAMA_MODEL)
        print(f"✅ {LLAMA_OLLAMA_MODEL} (Ollama) listo — sin token HuggingFace.")
    except Exception as e:
        print(f"❌ Error Ollama: {e}. Verifica que el servidor esté corriendo (Celda 4).")

def unload_llama():
    """Descarga llama3 de la VRAM de Ollama."""
    try:
        import requests as _req
        _req.post(
            "http://localhost:11434/api/generate",
            json={"model": LLAMA_OLLAMA_MODEL, "prompt": "", "keep_alive": 0},
            timeout=10,
        )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"♻️ {LLAMA_OLLAMA_MODEL} descargado de VRAM (Ollama).")
    except Exception as e:
        print(f"⚠️ No se pudo descargar Ollama de VRAM: {e}")

def load_presidio_bert():
    print(f"⬇️ Cargando Presidio + BERT ({BERT_MODEL})...")
    model_config = [{
        "lang_code": "es",
        "model_name": {"transformers": BERT_MODEL, "spacy": SPACY_MODEL},
    }]
    nlp_engine = TransformersNlpEngine(models=model_config)
    analyzer = AnalyzerEngine(
        nlp_engine=nlp_engine,
        default_score_threshold=PRESIDIO_SCORE_THRESHOLD,
    )
    anonymizer = AnonymizerEngine()
    print("✅ Presidio + BERT listos.")
    return analyzer, anonymizer

_TRANSLATORS = {}

def _load_translator(src, tgt):
    key = f"{src}-{tgt}"
    if key in _TRANSLATORS:
        return _TRANSLATORS[key]
    model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    try:
        tok = MarianTokenizer.from_pretrained(model_name)
        mdl = MarianMTModel.from_pretrained(model_name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        mdl = mdl.to(device).eval()
        _TRANSLATORS[key] = (tok, mdl)
        print(f" ✅ Traductor {key} cargado.")
        return tok, mdl
    except Exception as e:
        print(f" ⚠️ {model_name} no disponible: {str(e)[:60]}")
        _TRANSLATORS[key] = None
        return None

def preload_translators():
    print("⬇️ Pre-cargando traductores Helsinki-NLP...")
    for src, tgt in [("es","fr"),("en","es"),("ca","es"),("it","es"),("fr","es"),("gl","es")]:
        _load_translator(src, tgt)


# ═══════════════════════════════════════════════════════════════
# STAGE 1 — EXTRACCIÓN POR KEYWORDS
# ═══════════════════════════════════════════════════════════════

def _fetch_tweets_api(query):
    url = "https://api.twitterapi.io/twitter/tweet/advanced_search"
    resp = requests.get(url, headers={"x-api-key": TWITTER_API_KEY}, params={"query": query})
    resp.raise_for_status()
    return resp.json().get("tweets", [])

def _build_keyword_query(keywords, start_date, end_date):
    q = "(" + " OR ".join(f'"{k}"' for k in keywords) + ")"
    q += f" lang:{LANG} since:{start_date} until:{end_date}"
    q += f" geocode:{LAT},{LON},{RADIUS}"
    return q

def _parse_keyword_tweets(tweets, batch_keywords):
    rows = []
    for t in tweets:
        a = t.get("author", {})
        rows.append({
            "text": t.get("text"),
            "createdAt": t.get("createdAt"),
            "id": t.get("id"),
            "user_id": a.get("id"),
            "userName": a.get("userName"),
            "name": a.get("name"),
            "description": a.get("description"),
            "profile_bio": (a.get("profile_bio") or {}).get("description"),
            "location": a.get("location"),
            "followers": a.get("followers"),
            "following": a.get("following"),
            "retweets": t.get("retweetCount"),
            "replies": t.get("replyCount"),
            "likes": t.get("likeCount"),
            "quotes": t.get("quoteCount"),
            "views": t.get("viewCount"),
            "lang": t.get("lang"),
            "keywords": ", ".join(batch_keywords),
        })
    return rows

def _daily_ranges(start_year, end_year):
    cur = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    out = []
    while cur <= end:
        nxt = cur + timedelta(days=1)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return out

def _get_batches(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def stage1_extract_keyword_tweets():
    print("\n" + "="*60)
    print("FASE A · STAGE 1 — Extracción por keywords")
    print("="*60)
    if DEBUG_MODE:
        date_ranges = [
            (d, (datetime.strptime(d, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d"))
            for d in DEBUG_SPECIFIC_DATES
        ]
        keywords_to_use = DEBUG_KEYWORDS
        print(f"🧪 MODO DEBUG: {len(date_ranges)} fechas · {len(keywords_to_use)} keywords")
    else:
        date_ranges = _daily_ranges(START_YEAR, END_YEAR)
        keywords_to_use = KEYWORDS
    all_parsed = []

    for i, (s, e) in enumerate(date_ranges, 1):
        print(f" ({i}/{len(date_ranges)}) {s}", end=" ")
        day_found = 0
        for batch in _get_batches(keywords_to_use, KEYWORD_BATCH_SIZE):
            try:
                tweets = _fetch_tweets_api(_build_keyword_query(batch, s, e))
                if tweets:
                    all_parsed.extend(_parse_keyword_tweets(tweets, batch))
                    day_found += len(tweets)
            except Exception as ex:
                print(f"[err: {ex}]", end=" ")
            time.sleep(API_WAIT_TIME)
        print(f"→ {day_found} tweets")

    df = pd.DataFrame(all_parsed).drop_duplicates(subset=["id"])
    print(f"\n✅ Stage 1: {len(df)} tweets únicos · {df['user_id'].nunique()} usuarios únicos.")
    return df


# ═══════════════════════════════════════════════════════════════
# STAGE 2 — CLASIFICACIÓN Org/Indiv (Gemma 3 12B)
# ═══════════════════════════════════════════════════════════════

_CLASSIFY_PROMPT = """\
Clasifica este perfil de Twitter:
- "Org": empresa, institución, medio, ONG, marca o proyecto.
- "Indiv": persona individual.

Responde SOLO con JSON: {{"label": "Org" o "Indiv", "reason": "breve motivo"}}

Ejemplos:
Username: CruzRoja | Bio: Organización humanitaria → {{"label":"Org","reason":"org humanitaria"}}
Username: juan_23 | Bio: estudiante de ingeniería  → {{"label":"Indiv","reason":"perfil personal"}}

Clasifica:
Username: {userName}
Name: {name}
Bio: {bio}
Followers: {followers} | Following: {following}
"""

def _extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r'\{.*?\}', text, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except: pass
    return {"label": "Unknown", "reason": text[:80]}

def _normalize_label(label):
    l = str(label).lower()
    if "org"   in l: return "Org"
    if "indiv" in l: return "Indiv"
    return "Unknown"

@torch.inference_mode()
def _classify_with_gemma(row, model, processor):
    prompt = _CLASSIFY_PROMPT.format(
        userName = str(row.get("userName", "")),
        name = str(row.get("name", "")),
        bio = str(row.get("profile_bio", "") or "")[:300],
        followers = row.get("followers", ""),
        following = row.get("following", ""),
    )
    msgs = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    n_in = inputs["input_ids"].shape[-1]
    out = model.generate(
        **inputs, max_new_tokens=80, do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    resp = processor.decode(out[0][n_in:], skip_special_tokens=True).strip()
    parsed = _extract_json(resp)
    return _normalize_label(parsed.get("label", "Unknown"))

def stage2_classify_users(df_kw, gemma_model, gemma_processor):
    print("\n" + "="*60)
    print("FASE A · STAGE 2 — Clasificación Org / Indiv (Gemma 3 12B)")
    print("="*60)
    df_uniq = (
        df_kw.sort_values("createdAt", ascending=False)
             .drop_duplicates(subset="user_id").copy()
    )
    print(f"Usuarios únicos a clasificar: {len(df_uniq)}")

    labels = []
    for _, row in tqdm(df_uniq.iterrows(), total=len(df_uniq), desc="Clasificando"):
        try:
            lbl = _classify_with_gemma(row, gemma_model, gemma_processor)
        except Exception as e:
            lbl = "Unknown"
            print(f" ⚠️ {row.get('userName','?')}: {e}")
        labels.append(lbl)

    df_uniq["label"] = labels
    n_indiv = (df_uniq["label"] == "Indiv").sum()
    n_org = (df_uniq["label"] == "Org").sum()
    print(f"\n✅ Stage 2: {n_indiv} personas · {n_org} organizaciones.")

    indiv_usernames = (
        df_uniq[df_uniq["label"] == "Indiv"]["userName"]
        .dropna().unique().tolist()
    )
    return indiv_usernames


# ═══════════════════════════════════════════════════════════════
# STAGE 3 — FETCH TWEETS POR USUARIO
# ═══════════════════════════════════════════════════════════════

def _fetch_user_tweets_api(username):
    url = "https://api.twitterapi.io/twitter/user/last_tweets"
    headers = {"X-API-Key": TWITTER_API_KEY}
    tweets, seen, cursor = [], set(), None

    while len(tweets) < MAX_TWEETS_USER:
        params = {"userName": username}
        if cursor:
            params["cursor"] = cursor
        for attempt in range(3):
            try:
                r = requests.get(url, headers=headers, params=params)
                r.raise_for_status()
                data = r.json()
                break
            except Exception:
                time.sleep(2 ** attempt)
        else:
            break

        for t in data.get("data", {}).get("tweets", []):
            if t["id"] not in seen:
                seen.add(t["id"])
                tweets.append(t)
            if len(tweets) >= MAX_TWEETS_USER:
                break

        if not data.get("has_next_page", False):
            break
        cursor = data.get("next_cursor")

    return tweets

def _parse_user_tweets(raw_tweets):
    rows = []
    for t in raw_tweets:
        a = t.get("author", {})
        rows.append({
            "id": t.get("id"),
            "user_id": a.get("id"),
            "userName": a.get("userName"),
            "name": a.get("name"),
            "text": t.get("text"),
            "createdAt": t.get("createdAt"),
            "description": a.get("description"),
            "profile_bio": (a.get("profile_bio") or {}).get("description"),
            "location": a.get("location"),
            "followers": a.get("followers"),
            "following": a.get("following"),
            "retweets": t.get("retweetCount"),
            "replies": t.get("replyCount"),
            "likes": t.get("likeCount"),
            "quotes": t.get("quoteCount"),
            "views": t.get("viewCount"),
            "lang": t.get("lang"),
        })
    return rows


# ═══════════════════════════════════════════════════════════════
# STAGE 4 — ANONIMIZACIÓN MULTICAPA
# ═══════════════════════════════════════════════════════════════

_LABEL_MAP = {
    "PERSONA":   "[PERSONA]",   "PERSON":       "[PERSONA]",
    "UBICACION": "[UBICACION]", "LOCATION":     "[UBICACION]",
    "TELEFONO":  "[TELEFONO]",  "PHONE_NUMBER": "[TELEFONO]",
    "EMAIL":     "[EMAIL]",     "EMAIL_ADDRESS":"[EMAIL]",
    "DOCUMENTO": "[DOCUMENTO]",
}

def _hash_id(raw, salt):
    if not raw: return None
    return hashlib.sha256((str(raw) + salt).encode()).hexdigest()[:16]

def _hash_username(raw, salt):
    if not raw: return "USR_UNKNOWN"
    h = hashlib.sha256((str(raw).lower().strip().replace("@","") + salt).encode()).hexdigest()[:8]
    return f"USR_{h}"

def _simplify_date(d):
    if not d or not isinstance(d, str): return d
    try:
        return datetime.strptime(d, '%a %b %d %H:%M:%S %z %Y').strftime('%Y-%m-%d')
    except Exception:
        return d

# 4a — Regex
def _apply_regex(text):
    if not text or not isinstance(text, str): return text
    text = re.sub(r'\b[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'\b\d{7,}\b', '[NUMERO]', text)
    text = re.sub(r'@\w{2,}', '[USUARIO]', text)
    return text

# 4b — Presidio + BERT
def _apply_presidio(text, analyzer, anonymizer):
    if not text or not isinstance(text, str): return text, []
    try:
        results = analyzer.analyze(
            text=text,
            entities=["PERSON","LOCATION","PHONE_NUMBER","EMAIL_ADDRESS"],
            language="es",
        )
        if not results: return text, []
        anon = anonymizer.anonymize(text=text, analyzer_results=results)
        return anon.text, [{"entity": r.entity_type, "score": r.score} for r in results]
    except Exception:
        return text, []

# 4c — LLaMA 3 8B via Ollama 
_LLM_PII_PROMPT = (
    "Eres un detector de datos personales bajo RGPD para investigación en salud mental.\n"
    "Detecta PII residual: nombres propios, apellidos, lugares específicos, teléfonos, emails.\n"
    "Responde SOLO con JSON array. Si no hay nada: []\n"
    'Formato: [{{"texto":"valor","tipo":"PERSONA|UBICACION|TELEFONO|EMAIL|DOCUMENTO"}}]\n\n'
    'TEXTO: "{text}"'
)

def _detect_pii_llama(text):
    """Detecta PII residual con LLaMA 3 via Ollama (sin token HuggingFace)."""
    if not text or not isinstance(text, str): return []
    try:
        import ollama as _ollama
        prompt = _LLM_PII_PROMPT.format(text=text[:600])
        response = _ollama.generate(model=LLAMA_OLLAMA_MODEL, prompt=prompt)
        resp_text = response["response"].strip()
        m = re.search(r'\[.*?\]', resp_text, re.DOTALL)
        return json.loads(m.group()) if m else []
    except Exception:
        return []

def _apply_llm_detections(text, detections):
    for item in sorted(detections, key=lambda x: len(x.get("texto","")), reverse=True):
        label = _LABEL_MAP.get(item.get("tipo","").upper(), "[REDACTED]")
        text  = re.sub(re.escape(item["texto"]), label, text, flags=re.IGNORECASE)
    return text

def anonymize_tweet(tweet_dict, presidio_analyzer, presidio_anonymizer, salt):
    """
    Pipeline Stage 4 completo para un tweet.
    4d → 4a → 4b → 4c selectivo via Ollama (llama3)
    """
    t = copy.deepcopy(tweet_dict)

    # 4d — Hash identificadores
    t["id"] = _hash_id(t.get("id"), salt)
    t["user_id"] = _hash_id(t.get("user_id"), salt)
    t["userName"] = _hash_username(t.get("userName"), salt)
    t["name"] = "[NOMBRE]" if t.get("name") else None
    t["createdAt"] = _simplify_date(str(t.get("createdAt", "")))

    # Campos de texto
    for field in ["text", "description", "profile_bio", "location"]:
        val = t.get(field)
        if not val or not isinstance(val, str) or not val.strip():
            continue

        # 4a Regex
        val = _apply_regex(val)

        # 4b Presidio + BERT
        val, hits = _apply_presidio(val, presidio_analyzer, presidio_anonymizer)

        # 4c LLaMA — selectivo
        run_llm = (not LLM_ONLY_WITH_ENTITIES) or bool(hits) or (field == "profile_bio")
        if run_llm:
            llm_hits = _detect_pii_llama(val)
            if llm_hits:
                val = _apply_llm_detections(val, llm_hits)

        t[field] = val

    return t


# ═══════════════════════════════════════════════════════════════
# STAGE 5 — GENERALIZACIÓN profile_bio (Gemma 3 12B + Helsinki)
# ═══════════════════════════════════════════════════════════════

_SUPPORTED_TO_ES = {"ca","en","it","fr","gl"}

def _norm_unicode(text):
    return unicodedata.normalize("NFKC", text) if isinstance(text, str) else ""

def _detect_lang(text):
    if not text or len(text.strip()) < 5: return "unknown"
    try: return detect(text)
    except LangDetectException: return "unknown"

@torch.inference_mode()
def _translate(text, src, tgt):
    if not text or not text.strip(): return text
    pair = _load_translator(src, tgt)
    if pair is None: return text
    tok, mdl = pair
    try:
        inp = tok(text, return_tensors="pt", truncation=True, max_length=512).to(mdl.device)
        out = mdl.generate(**inp, max_length=512, num_beams=4, early_stopping=True)
        return tok.decode(out[0], skip_special_tokens=True)
    except Exception:
        return text

def _ensure_spanish(text):
    lang = _detect_lang(text)
    if lang in ("es","unknown"): return text, lang
    if lang in _SUPPORTED_TO_ES: return _translate(text, lang, "es"), lang
    return text, lang

_GENERALIZE_PROMPT = """\
Eres un anonimizador de perfiles de Twitter para investigación en salud mental.

ELIMINA obligatoriamente:
- Nombres propios de personas, organizaciones específicas, ubicaciones específicas
- Handles de redes sociales, fechas exactas
- Placeholders residuales: <LOCATION> <PERSON> <PHONE_NUMBER> [USUARIO] [URL] [EMAIL] [NUMERO] [NOMBRE]

PRESERVA:
- Género gramatical (psicóloga vs psicólogo, She/Her, He/Him)
- Roles familiares genéricos (madre, padre, hijo)
- Categoría profesional genérica (sanitario, jurista, docente)
- Indicadores de edad aproximada (23 años, generación Z, desde los 90)
- Región amplia (España, Latinoamérica)
- Temas de interés generales (música, deporte, salud mental)

Si es solo emojis o frase poética corta sin información personal: devuélvela tal cual.

BIO A GENERALIZAR: "{bio}"
GENERALIZADA:"""

@torch.inference_mode()
def _generalize_bio(bio, model, processor):
    if not bio or not bio.strip(): return bio
    prompt = _GENERALIZE_PROMPT.replace("{bio}", bio[:400])
    msgs   = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    n_in = inputs["input_ids"].shape[-1]
    out  = model.generate(
        **inputs, max_new_tokens=200, do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    resp = processor.decode(out[0][n_in:], skip_special_tokens=True).strip()
    resp = re.sub(r'^(GENERALIZADA|GEN)[:\s]*', '', resp, flags=re.IGNORECASE).strip('\"\' ')
    lines = [l.strip() for l in resp.split("\n") if l.strip()]
    return lines[0] if lines else bio

def process_bio_stage5(anon_bio, gemma_model, gemma_processor):
    """Stage 5 completo para una bio ya anonimizada por Stage 4."""
    if anon_bio is None or (isinstance(anon_bio, float) and pd.isna(anon_bio)):
        return ""
    if not isinstance(anon_bio, str) or anon_bio.strip().lower() in ("nan","none",""):
        return ""

    bio = _norm_unicode(anon_bio)
    if len(bio.strip()) < 3: return bio

    bio_es, _ = _ensure_spanish(bio)

    try:
        bio_gen = _generalize_bio(bio_es, gemma_model, gemma_processor)
    except Exception as e:
        print(f"⚠️ Error LLM en bio: {e}")
        bio_gen = bio_es

    try:
        bio_fr = _translate(bio_gen, "es", "fr")
    except Exception:
        bio_fr = bio_gen

    return bio_fr


# ═══════════════════════════════════════════════════════════════
# CHECKPOINT UTILS
# ═══════════════════════════════════════════════════════════════

def _append_checkpoint(records, path):
    """Append anonymized tweet records to a JSONL file in Drive."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def _load_processed_users(path):
    """Return set of source_username already in checkpoint."""
    if not os.path.exists(path): return set()
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try: done.add(json.loads(line).get("_src_user",""))
            except: pass
    return done

print("✅ Todas las funciones definidas.")

In [ ]:
# Montar Google Drive (para checkpoints opcionales)
from google.colab import drive
drive.mount('/content/drive')

# Gemma 3 12B (Fase A: Stage 2 · Fase C: Stage 5)
gemma_model, gemma_processor = load_gemma()

# Presidio + BERT (Stage 4b · permanece cargado)
presidio_analyzer, presidio_anonymizer = load_presidio_bert()

# Traductores Helsinki-NLP (Stage 5 · permanecen cargados)
preload_translators()

# Servidor Ollama (para LLaMA 3 en Fase B)
# Inicia el servidor en segundo plano; el modelo se carga en GPU
# solo cuando se hace la primera llamada (Fase B).
import subprocess, threading, time

def _run_ollama_server():
    subprocess.run(['ollama', 'serve'], capture_output=True)

_ollama_thread = threading.Thread(target=_run_ollama_server, daemon=True)
_ollama_thread.start()
time.sleep(4) # esperar a que el servidor arranque

# Pre-descargar el modelo (solo la primera vez, ~4.7 GB)
print("⬇️ Verificando / descargando llama3 via Ollama...")
subprocess.run(['ollama', 'pull', 'llama3'], check=True)
print("✅ llama3 disponible. Se cargará en GPU al inicio de la Fase B.")

print("\n✅ Todos los modelos listos. Ejecuta la Celda 5 para lanzar el pipeline.")


In [ ]:
def run_pipeline():
    """
    Ejecuta el pipeline completo:
      Fase A : Stage 1 (keywords) + Stage 2 (clasificación Gemma)
      SWAP 1 : descarga Gemma HF → Ollama carga llama3 en primera inferencia
      Fase B : Loop por usuario → Stage 3 (fetch) + Stage 4 (anonimización)
      SWAP 2 : descarga llama3 de Ollama → recarga Gemma HF
      Fase C : Stage 5 (generalización bio + traducción FR)
    Retorna df_final (DataFrame anonimizado en memoria).
    """
    global gemma_model, gemma_processor

    print("\n" + "🚀"*20)
    print("PIPELINE UNIFICADO TFM — INICIO")
    print("🚀"*20)

    # ───────────────────────────────────────────────────────────
    # FASE A — Stage 1 + Stage 2 (Gemma 3 12B)
    # ───────────────────────────────────────────────────────────
    df_kw = stage1_extract_keyword_tweets()

    indiv_usernames = stage2_classify_users(df_kw, gemma_model, gemma_processor)

    # Liberar tweets keyword (ya no necesarios; solo conservamos usernames)
    del df_kw
    gc.collect()

    # Aplicar offset de sesión (para reanudar)
    if SESSION_OFFSET > 0:
        print(f"\n⏩ SESSION_OFFSET={SESSION_OFFSET}: saltando los primeros {SESSION_OFFSET} usuarios.")
        indiv_usernames = indiv_usernames[SESSION_OFFSET:]

    # Saltar usuarios ya procesados en checkpoint
    if USE_DRIVE_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
        already_done = _load_processed_users(CHECKPOINT_PATH)
        n_before = len(indiv_usernames)
        indiv_usernames = [u for u in indiv_usernames if u not in already_done]
        print(f"🔄 Checkpoint: {n_before - len(indiv_usernames)} ya procesados, {len(indiv_usernames)} pendientes.")

    if not indiv_usernames:
        print("✅ Todos los usuarios ya fueron procesados. Reconstruyendo df_final desde checkpoint...")

    # Modo debug (limita nº de usuarios para prueba rápida)
    if MAX_USERS_DEBUG:
        print(f"\n🧪 MODO DEBUG: procesando solo {MAX_USERS_DEBUG} usuarios de {len(indiv_usernames)} disponibles.")
        indiv_usernames = indiv_usernames[:MAX_USERS_DEBUG]

    # ───────────────────────────────────────────────────────────
    # SWAP 1 — Gemma → LLaMA 3 8B
    # ───────────────────────────────────────────────────────────
    if SWAP_MODELS:
        print("\n🔄 SWAP 1: descargando Gemma 3 12B de VRAM...")
        unload_model(gemma_model, "Gemma 3 12B")
        gemma_model = None
        gemma_processor = None

    # llama3 se cargará automáticamente en GPU en la primera llamada Ollama
    load_llama()

    # ───────────────────────────────────────────────────────────
    # FASE B — Loop por usuario (Stage 3 + Stage 4)
    # ───────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"FASE B — Loop por usuario ({len(indiv_usernames)} pendientes)")
    print("="*60)

    df_accumulator = [] # tweets anonimizados de la sesión actual
    user_bios = {} # user_id_hash → anon_profile_bio (para Stage 5)
    n_ok = 0
    n_err = 0

    for username in tqdm(indiv_usernames, desc="Usuarios"):
        try:
            # Stage 3: Descargar tweets del usuario
            raw = _fetch_user_tweets_api(username)
            if not raw:
                print(f"⚠️ {username}: sin tweets.")
                continue

            tweet_dicts = _parse_user_tweets(raw)

            # Stage 4: Anonimizar tweet a tweet
            anon_list = []
            for td in tweet_dicts:
                anon = anonymize_tweet(
                    td,
                    presidio_analyzer, presidio_anonymizer,
                    VOLATILE_SALT,
                )
                anon["_src_user"] = username # tracking para checkpoint
                anon_list.append(anon)

            # Guardar bio anonimizada para Stage 5 (primera no vacía del usuario)
            uid_hash = _hash_id(tweet_dicts[0].get("user_id"), VOLATILE_SALT)
            if uid_hash and uid_hash not in user_bios:
                for anon in anon_list:
                    bio = anon.get("profile_bio")
                    if bio and isinstance(bio, str) and bio.strip():
                        user_bios[uid_hash] = bio
                        break

            # Acumular en RAM
            df_accumulator.extend(anon_list)

            # Checkpoint opcional en Drive (ya son datos anonimizados → RGPD ok)
            if USE_DRIVE_CHECKPOINT:
                _append_checkpoint(anon_list, CHECKPOINT_PATH)

            # Limpiar datos crudos de RAM
            del raw, tweet_dicts, anon_list
            gc.collect()
            n_ok += 1

        except Exception as e:
            print(f"❌ Error con {username}: {e}")
            n_err += 1

    print(f"\n✅ Fase B completa: {n_ok} usuarios procesados · {n_err} errores.")
    print(f" Bios únicas recopiladas para Stage 5: {len(user_bios)}")

    # ───────────────────────────────────────────────────────────
    # SWAP 2 — LLaMA → Gemma 3 12B
    # ───────────────────────────────────────────────────────────
    if SWAP_MODELS:
        print("\n🔄 SWAP 2: descargando llama3 de VRAM (Ollama)...")
        unload_llama()

    gemma_model, gemma_processor = load_gemma()

    # ───────────────────────────────────────────────────────────
    # FASE C — Stage 5: Generalización bio (Gemma + Helsinki)
    # ───────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"FASE C · STAGE 5 — Generalización profile_bio ({len(user_bios)} bios únicas)")
    print("="*60)

    bio_fr_map = {}
    for uid_hash, anon_bio in tqdm(user_bios.items(), desc="Generalizando bios"):
        bio_fr_map[uid_hash] = process_bio_stage5(anon_bio, gemma_model, gemma_processor)

    # ───────────────────────────────────────────────────────────
    # CONSTRUCCIÓN DE df_final
    # ───────────────────────────────────────────────────────────
    print("\n📦 Construyendo df_final...")

    # Combinar sesión actual + checkpoint anterior (si existe)
    all_records = list(df_accumulator)

    if USE_DRIVE_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
        # Cargar registros de sesiones anteriores que no están en la sesión actual
        current_src_users = {r.get("_src_user","") for r in all_records}
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get("_src_user","") not in current_src_users:
                        all_records.append(rec)
                except Exception:
                    pass

    df_final = pd.DataFrame(all_records)

    # Añadir profile_bio_fr
    if not df_final.empty and "user_id" in df_final.columns:
        df_final["profile_bio_fr"] = df_final["user_id"].map(bio_fr_map).fillna("")

    # Eliminar columna de tracking interna
    if "_src_user" in df_final.columns:
        df_final = df_final.drop(columns=["_src_user"])

    # ───────────────────────────────────────────────────────────
    # RESUMEN FINAL
    # ───────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("🏁 PIPELINE COMPLETADO")
    print("="*60)
    print(f"Tweets anonimizados totales: {len(df_final):,}")
    if not df_final.empty:
        print(f"Usuarios únicos (hasheados): {df_final['user_id'].nunique():,}")
        print(f"Bios generalizadas en FR: {(df_final['profile_bio_fr'] != '').sum():,}")
    print(f"Columnas: {list(df_final.columns)}")
    print("\n ✅ df_final disponible en memoria. Sin PII. RGPD compliant.")
    print("="*60)

    return df_final


# EJECUCIÓN
df_final = run_pipeline()